# YOLOv11n-cls Multi-Seed Training

Stable training for YOLOv11n-cls (Edge Model, ~3MB) on Rice_Leaf_AUG.
**Trains on multiple seeds and reports the best results.**

Stability features:
1. Multiple seeds for robust evaluation
2. Cosine LR schedule — smooth single-cycle decay
3. Deterministic mode — ensures reproducibility within a seed

## Environment Setup

In [ ]:
!pip install split-folders ultralytics

In [ ]:
import os
import sys

# Set environment variables for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Check if torch was already imported (common on Kaggle)
if 'torch' in sys.modules:
    print("⚠️  WARNING: torch was already imported by Kaggle.")
    print("   For strict determinism, add CUBLAS_WORKSPACE_CONFIG=:4096:8")
    print("   in Kaggle Settings → Add-ons → Environment Variables")
    print("   Then restart kernel and run all cells.")
    print("   Continuing with warn_only=True mode...")

## Imports

In [ ]:
import random
import numpy as np
import splitfolders
import torch
import shutil
import json
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

## Configuration

In [ ]:
# ============================================
# MULTI-SEED CONFIGURATION
# ============================================
SEEDS = [1, 42, 123, 456, 789, 1024]  # List of seeds to try

# Data configuration
SPLIT_SEED      = 42              # Keep fixed for fair comparison across seeds
DATA_DIR        = "/kaggle/input/datasets/anshulm257/rice-disease-dataset/Rice_Leaf_AUG"
SPLIT_DIR       = "data/splits"

# Training hyperparameters
EPOCHS          = 150
BATCH_SIZE      = 32
IMG_SIZE        = 320
PATIENCE        = 25
LABEL_SMOOTHING = 0.05
WEIGHT_DECAY    = 5e-4
LR0             = 1e-3
LRF             = 0.001
WARMUP_EPOCHS   = 3
DROPOUT         = 0.1
MIXUP           = 0.15
COPY_PASTE      = 0.1

# Output
OUTPUT_DIR      = "outputs"
GPUS            = ""

print(f"Will train with {len(SEEDS)} seeds: {SEEDS}")

## Helper Functions

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
    # Check if CUBLAS_WORKSPACE_CONFIG is properly set
    cublas_config = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    if cublas_config in [":4096:8", ":16:8"]:
        torch.use_deterministic_algorithms(True, warn_only=False)
    else:
        torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
def setup_split(data_dir, split_dir, split_seed):
    if (not os.path.exists(split_dir)
            or not os.path.exists(os.path.join(split_dir, "train"))
            or not os.path.exists(os.path.join(split_dir, "test"))):
        print(f"Splitting '{data_dir}' -> 70/15/15 train/val/test (split_seed={split_seed}) ...")
        splitfolders.ratio(data_dir, output=split_dir, seed=split_seed, ratio=(0.70, 0.15, 0.15))
        print(f"Split saved to '{split_dir}'")
    else:
        print(f"Using existing split in '{split_dir}'")
    return split_dir

In [ ]:
def train_single_seed(seed, split_dir, device, seed_output_dir):
    """Train YOLO with a single seed and return results."""
    print(f"\n{'='*60}")
    print(f"  Training with SEED: {seed}")
    print(f"{'='*60}")
    
    seed_everything(seed)
    os.makedirs(seed_output_dir, exist_ok=True)
    
    model = YOLO("yolo11n-cls.pt")
    results = model.train(
        data            = split_dir,
        epochs          = EPOCHS,
        batch           = BATCH_SIZE,
        imgsz           = IMG_SIZE,
        project         = seed_output_dir,
        name            = f"seed_{seed}",
        exist_ok        = True,
        device          = device,
        seed            = seed,
        deterministic   = True,
        cos_lr          = True,
        dropout         = DROPOUT,
        weight_decay    = WEIGHT_DECAY,
        label_smoothing = LABEL_SMOOTHING,
        mixup           = MIXUP,
        copy_paste      = COPY_PASTE,
        degrees         = 10.0,
        flipud          = 0.1,
        fliplr          = 0.5,
        hsv_h           = 0.015,
        hsv_s           = 0.5,
        hsv_v           = 0.3,
        warmup_epochs   = WARMUP_EPOCHS,
        lr0             = LR0,
        lrf             = LRF,
        patience        = PATIENCE,
        verbose         = True,
    )
    
    best_pt = results.save_dir / "weights" / "best.pt"
    
    # Test evaluation
    top1, top5 = 0.0, 0.0
    if best_pt.exists():
        tmodel = YOLO(str(best_pt))
        tres = tmodel.val(data=split_dir, split="test",
                         imgsz=IMG_SIZE, batch=BATCH_SIZE, device=device)
        top1 = float(tres.results_dict.get("metrics/accuracy_top1", 0))
        top5 = float(tres.results_dict.get("metrics/accuracy_top5", 0))
    
    print(f"\n📊 Seed {seed} →  Top-1: {top1*100:.2f}%  |  Top-5: {top5*100:.2f}%")
    
    return {
        "seed": seed,
        "top1_acc": top1 * 100,
        "top5_acc": top5 * 100,
        "best_pt": str(best_pt),
        "save_dir": str(results.save_dir),
    }

## Setup Data Split

In [ ]:
split_dir = setup_split(DATA_DIR, SPLIT_DIR, SPLIT_SEED)
device = GPUS if GPUS else (0 if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nDevice: {device}")
print(f"Data split: {split_dir}")

## Multi-Seed Training Loop

In [ ]:
all_results = []

print(f"\n{'#'*60}")
print(f"  STARTING MULTI-SEED TRAINING")
print(f"  Seeds: {SEEDS}")
print(f"{'#'*60}")

for i, seed in enumerate(SEEDS):
    print(f"\n[{i+1}/{len(SEEDS)}] Training with seed {seed}...")
    seed_output_dir = os.path.join(OUTPUT_DIR, "runs")
    
    result = train_single_seed(seed, split_dir, device, seed_output_dir)
    all_results.append(result)
    
    # Save intermediate results
    with open(os.path.join(OUTPUT_DIR, "all_seed_results.json"), "w") as f:
        json.dump(all_results, f, indent=2)

print(f"\n{'#'*60}")
print(f"  MULTI-SEED TRAINING COMPLETE")
print(f"{'#'*60}")

## Results Comparison

In [ ]:
# Create results dataframe
df = pd.DataFrame(all_results)
df = df.sort_values("top1_acc", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("  RESULTS SUMMARY (Sorted by Top-1 Accuracy)")
print("="*70)
print(df[["seed", "top1_acc", "top5_acc"]].to_string(index=False))
print("="*70)

# Statistics
print(f"\n📊 Statistics:")
print(f"   Top-1 Accuracy: Mean={df['top1_acc'].mean():.2f}%, Std={df['top1_acc'].std():.2f}%")
print(f"   Top-5 Accuracy: Mean={df['top5_acc'].mean():.2f}%, Std={df['top5_acc'].std():.2f}%")
print(f"   Best seed: {df.iloc[0]['seed']} with Top-1: {df.iloc[0]['top1_acc']:.2f}%")
print(f"   Worst seed: {df.iloc[-1]['seed']} with Top-1: {df.iloc[-1]['top1_acc']:.2f}%")

## Best Model Export

In [ ]:
# Get best result
best_result = df.iloc[0]
best_seed = int(best_result["seed"])
best_pt = Path(best_result["best_pt"])
best_save_dir = Path(best_result["save_dir"])

print(f"\n🏆 BEST MODEL")
print(f"   Seed: {best_seed}")
print(f"   Top-1 Accuracy: {best_result['top1_acc']:.2f}%")
print(f"   Top-5 Accuracy: {best_result['top5_acc']:.2f}%")
print(f"   Model path: {best_pt}")

# Export best model to ONNX
if best_pt.exists():
    print("\n📦 Exporting best model to ONNX ...")
    onnx_model = YOLO(str(best_pt))
    onnx_path = onnx_model.export(format="onnx", dynamic=True)
    print(f"✅ ONNX model → {onnx_path}")
    
    # Copy best model to output directory
    best_output_dir = os.path.join(OUTPUT_DIR, "best_model")
    os.makedirs(best_output_dir, exist_ok=True)
    shutil.copy2(best_pt, os.path.join(best_output_dir, "best.pt"))
    if onnx_path:
        shutil.copy2(onnx_path, os.path.join(best_output_dir, "best.onnx"))
    print(f"✅ Best model copied to: {best_output_dir}")

## Display Best Model Training Curves

In [ ]:
results_img = best_save_dir / "results.png"
if results_img.exists():
    print(f"📈 Training Results for Best Seed ({best_seed}):")
    display(Image(filename=str(results_img)))

## Display Best Model Confusion Matrix

In [ ]:
confusion_matrix_normalized_img = best_save_dir / "confusion_matrix_normalized.png"
confusion_matrix_img = best_save_dir / "confusion_matrix.png"

if confusion_matrix_normalized_img.exists():
    print(f"📊 Confusion Matrix (Normalized) for Best Seed ({best_seed}):")
    display(Image(filename=str(confusion_matrix_normalized_img)))
elif confusion_matrix_img.exists():
    print(f"📊 Confusion Matrix for Best Seed ({best_seed}):")
    display(Image(filename=str(confusion_matrix_img)))

## Save Final Summary

In [ ]:
# Save complete summary
summary = {
    "seeds_tested": SEEDS,
    "split_seed": SPLIT_SEED,
    "best_seed": best_seed,
    "best_top1_acc": float(best_result["top1_acc"]),
    "best_top5_acc": float(best_result["top5_acc"]),
    "mean_top1_acc": float(df["top1_acc"].mean()),
    "std_top1_acc": float(df["top1_acc"].std()),
    "mean_top5_acc": float(df["top5_acc"].mean()),
    "std_top5_acc": float(df["top5_acc"].std()),
    "all_results": all_results,
}

summary_file = os.path.join(OUTPUT_DIR, "multi_seed_summary.json")
with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2)

# Save CSV for easy viewing
csv_file = os.path.join(OUTPUT_DIR, "seed_results.csv")
df.to_csv(csv_file, index=False)

print(f"\n✅ Summary saved to: {summary_file}")
print(f"✅ CSV results saved to: {csv_file}")

## Final Summary

In [ ]:
print(f"\n{'='*70}")
print(f"  FINAL SUMMARY")
print(f"{'='*70}")
print(f"  Seeds tested     : {SEEDS}")
print(f"  Best seed        : {best_seed}")
print(f"  Best Top-1 Acc   : {best_result['top1_acc']:.2f}%")
print(f"  Best Top-5 Acc   : {best_result['top5_acc']:.2f}%")
print(f"  Mean Top-1 Acc   : {df['top1_acc'].mean():.2f}% (±{df['top1_acc'].std():.2f}%)")
print(f"  Best model       : {os.path.join(OUTPUT_DIR, 'best_model', 'best.pt')}")
print(f"  ONNX model       : {os.path.join(OUTPUT_DIR, 'best_model', 'best.onnx')}")
print(f"{'='*70}")